# DSI Skeleton v2 — Squarepoint Capital

**Автор:** Sergey Nefedov
**Назначение:** Универсальный шаблон для Data Science Interview (DSI)
**Использование:** Копируй блоки, адаптируй под датасет. В markdown — теория и возможные вопросы интервьюера, чтобы не зависнуть.

---

## Структура

| # | Блок | Ключевые проверки |
|---|------|------------------|
| 1 | Imports & Config | Стек, seed, настройки отображения |
| 2 | Load & First Look | Shape, types, memory, schema |
| 3 | Data Quality | Missing patterns, duplicates, outliers, target distribution |
| 4 | EDA — Univariate | Distributions, skew, kurtosis, cardinality |
| 5 | EDA — Bivariate | Pearson/Spearman, VIF, scatter with trend |
| 6 | Feature Engineering | Yeo-Johnson, cross-sectional ranks, interactions, NO LEAKAGE |
| 7 | Preprocessing & Split | Pipeline, ColumnTransformer, temporal vs random split |
| 8 | Baseline | DummyRegressor/Classifier — sanity check |
| 9 | ML Models | RidgeCV, Lasso, ElasticNet, LightGBM, XGBoost |
| 10 | Validation | Walk-forward, Purged CV with embargo, learning curves |
| 11 | Quant Analysis | IC, ICIR (Newey-West), quintiles, turnover |
| 12 | Residual & Error Analysis | Residual plots, Q-Q, heteroscedasticity |
| 13 | Robustness Checks | Stability, sensitivity, multiple testing |
| 14 | Conclusions | Summary + next steps |

---

## Общие принципы DSI (что оценивают)

1. **Data cleaning decisions** — каждое решение (дроп/импут/винзоризация) нужно обосновать.
2. **No leakage** — train никогда не видит test, даже через статистики для FE.
3. **Baselines first** — всегда начинаем с простейшей модели как точки отсчёта.
4. **Validation matches reality** — для temporal данных только walk-forward / purged CV.
5. **Hypothesis testing > model complexity** — интервьюеры ценят правильные тесты, а не тюнинг гиперпараметров.
6. **Interpretability** — уметь объяснить каждый коэффициент и feature importance.


---
## 1. Imports & Configuration

Загружаем весь стек сразу. Интервьюер видит, что ты знаешь библиотеки.
Настраиваем pandas/matplotlib один раз глобально — меньше шума в ноутбуке.

**Возможный вопрос:** *"Зачем `np.random.seed()` + `random_state` в каждой модели?"*
→ **Воспроизводимость**. `np.seed` покрывает numpy; sklearn-модели используют свой RNG, поэтому им отдельно нужен `random_state`. LightGBM/XGBoost — тоже отдельные генераторы.


In [ ]:
# === CORE ===
import numpy as np
import pandas as pd

# === VISUALIZATION ===
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# === STATISTICS ===
from scipy import stats
from scipy.stats import spearmanr, pearsonr, normaltest, shapiro, jarque_bera

# Econometrics (для Newey-West / ADF / robust errors)
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.multitest import multipletests

# === SKLEARN — PREPROCESSING ===
from sklearn.preprocessing import (
    StandardScaler, RobustScaler, PowerTransformer,
    OneHotEncoder, OrdinalEncoder
)
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# === SKLEARN — MODELS ===
from sklearn.linear_model import (
    Ridge, RidgeCV, Lasso, LassoCV, ElasticNet, ElasticNetCV,
    LinearRegression, LogisticRegression, LogisticRegressionCV, HuberRegressor
)
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.dummy import DummyRegressor, DummyClassifier

# === SKLEARN — VALIDATION ===
from sklearn.model_selection import (
    train_test_split, cross_val_score,
    TimeSeriesSplit, KFold, StratifiedKFold,
    GridSearchCV, learning_curve
)
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, roc_auc_score, f1_score,
    classification_report, confusion_matrix
)

# === BOOSTING ===
import lightgbm as lgb
import xgboost as xgb

# === WARNINGS ===
import warnings
warnings.filterwarnings("ignore")

# === DISPLAY SETTINGS ===
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:.4f}".format)
pd.set_option("display.width", 1000)

# === PLOT SETTINGS ===
plt.rcParams["figure.figsize"]    = (12, 5)
plt.rcParams["axes.spines.top"]   = False
plt.rcParams["axes.spines.right"] = False
plt.rcParams["font.size"]         = 11
sns.set_palette("muted")

# === RANDOM SEED ===
SEED = 42
np.random.seed(SEED)

print("All imports OK")


---
## 2. Load & First Look

**Цель первых 5 минут:** понять структуру, не делая выводов.

Вопросы себе:
- Сколько строк/колонок? Это "широкий" датасет (p >> n) или "длинный"?
- Какие типы? Есть ли `object`, которые на самом деле числа/даты (грязь)?
- Что таргет? Регрессия/классификация/ранжирование?
- Есть ли временная ось? (определяет стратегию валидации)
- Есть ли groupby-измерение (asset_id, user_id)? — важно для leakage prevention

**Возможные вопросы:**
- *"Почему `memory_usage(deep=True)`?"* → без `deep=True` object-колонки считают только указатели, не реальные строки. Для датасета со строками разница может быть в 100x.
- *"Что если данные не помещаются в память?"* → chunked reading (`pd.read_csv(..., chunksize=...)`), Parquet + Arrow, Dask, или семплирование.


In [ ]:
# --- Загрузка ---
df = pd.read_csv("data.csv")
# df = pd.read_excel("data.xlsx")
# df = pd.read_parquet("data.parquet")   # Предпочтительный формат для больших файлов

print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Memory: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")


In [ ]:
# --- Первый взгляд ---
df.head(10)


In [ ]:
# --- Типы и non-null counts ---
# Ищем: object-колонки (возможно категории или грязные числа), datetime, float32 vs float64
df.info()


In [ ]:
# --- Базовая статистика ---
# Смотрим: min/max на адекватность (отрицательные цены?), mean vs median (скос), std
# Раздельно для numeric и object, иначе describe режет инфу
print("=== Numeric ===")
display(df.describe().T)

print("\n=== Object / Category ===")
obj_cols = df.select_dtypes(include=["object", "category"]).columns
if len(obj_cols) > 0:
    display(df[obj_cols].describe().T)
else:
    print("No object columns")


In [ ]:
# --- Имена колонок ---
print("Columns:")
for i, col in enumerate(df.columns):
    dtype = df[col].dtype
    n_unique = df[col].nunique()
    print(f"  {i:2d}. {col:30s} | {str(dtype):12s} | unique={n_unique}")


---
## 3. Data Quality

**Никогда не пропускай.** Грязные данные — причина #1 плохих моделей, и на DSI это проверяют в первую очередь.

### Четыре проверки:
1. **Missingness** — сколько, где, паттерн
2. **Duplicates** — полные и по ключу
3. **Types** — корректность типов (даты как строки, числа как object)
4. **Outliers** — аномалии в признаках и таргете

### Теория: типы пропусков (обязательно знать!)

| Тип | Расшифровка | Пример | Что делать |
|-----|-------------|--------|------------|
| **MCAR** | Missing Completely At Random | Случайные сбои сенсора | Drop или простой импут — безопасно |
| **MAR** | Missing At Random | Пропуски зависят от других *наблюдаемых* признаков | Model-based impute (KNN, IterativeImputer) |
| **MNAR** | Missing Not At Random | Пропуски зависят от *самого* пропущенного значения (богатые не указывают доход) | Сложно. Явный флаг `is_missing` + модель на флаге |

**Возможные вопросы:**
- *"Как тестировать MCAR?"* → Little's MCAR test, но на практике — просто сравниваем распределения признаков на `is_null` vs `not_null`.
- *"Почему не всегда дропать пропуски?"* → Теряем сигнал (MNAR случаи информативны), уменьшаем выборку.
- *"Median vs mean impute?"* → Median устойчив к выбросам. Всегда предпочтительнее для skewed признаков.


In [ ]:
# === ПРОПУСКИ ===
missing = pd.DataFrame({
    "count": df.isnull().sum(),
    "pct":   df.isnull().mean() * 100
}).query("count > 0").sort_values("pct", ascending=False)

print(f"Колонок с пропусками: {len(missing)} из {df.shape[1]}")
if len(missing) > 0:
    print(missing.to_string())

    # Визуализация
    fig, ax = plt.subplots(figsize=(10, max(4, len(missing) * 0.25)))
    missing["pct"].sort_values().plot(kind="barh", ax=ax, color="steelblue")
    ax.set_xlabel("% пропусков")
    ax.set_title("Пропуски по колонкам")
    plt.tight_layout()
    plt.show()
else:
    print("Пропусков нет.")


In [ ]:
# === ПАТТЕРН ПРОПУСКОВ (MCAR vs MAR hint) ===
# Если пропуски в колонке A коррелируют с пропусками в колонке B — это не случайность.
if len(missing) >= 2:
    null_matrix = df[missing.index].isnull().astype(int)
    null_corr = null_matrix.corr()

    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(null_corr, annot=True, fmt=".2f", cmap="RdBu_r",
                center=0, vmin=-1, vmax=1, ax=ax)
    ax.set_title("Корреляция паттернов пропусков")
    plt.tight_layout()
    plt.show()

    # Комментарий: высокая корреляция => пропуски связаны => вероятно MAR или MNAR
    high_corr = (null_corr.abs() > 0.5) & (null_corr.abs() < 1.0)
    if high_corr.any().any():
        print("\n⚠️  Обнаружены связанные пропуски (MAR/MNAR сигнал):")
        for c in null_corr.columns:
            for r in null_corr.index:
                if c < r and high_corr.loc[r, c]:
                    print(f"   {r} ↔ {c}: corr = {null_corr.loc[r, c]:.2f}")


In [ ]:
# === ДУБЛИКАТЫ ===
n_full = df.duplicated().sum()
print(f"Полных дубликатов: {n_full} ({n_full/len(df)*100:.2f}%)")

# Дубликаты по бизнес-ключу (ID) — обычно признак проблемы в ETL
# id_col = "id"
# n_id_dups = df[id_col].duplicated().sum()
# print(f"Дубликатов по {id_col}: {n_id_dups}")
# if n_id_dups > 0:
#     display(df[df[id_col].duplicated(keep=False)].sort_values(id_col).head(20))


### Outliers: какой метод выбрать?

| Метод | Когда применять | Плюсы | Минусы |
|-------|----------------|-------|--------|
| **IQR (1.5×)** | Классика Тьюки, нормальные распределения | Простой | Слишком агрессивен для heavy-tailed |
| **IQR (3×)** | Финансовые данные, тяжёлые хвосты | Консервативнее | Пропускает мягкие аномалии |
| **Z-score** | Нормальное распределение | Интуитивный | Ломается на heavy tails (std сам раздут выбросами) |
| **Modified Z (MAD)** | Robust альтернатива Z | Устойчив | Менее известен |
| **Isolation Forest** | Многомерные выбросы | Ловит "странные комбинации" | Чёрный ящик |

**Что делать с выбросами:**
- **Drop** — только если уверен, что это ошибки (человеческий ввод, битый сенсор)
- **Winsorize** (клиппинг к percentile, например 1% / 99%) — стандарт в квантах, сохраняет объекты
- **Log/Yeo-Johnson transform** — убирает "длину хвоста" без потери наблюдений
- **Robust модели** (HuberRegressor, Quantile Regression) — работает со всем
- **Оставить и использовать MAE/Huber loss** — лучший вариант, если выбросы реальны

**Вопрос-ловушка:** *"Надо ли винзорить таргет?"*
→ **Регрессия: чаще нет** (иначе модель не будет предсказывать экстремумы). **Квант факторы: да** (winsorize признаки на ±3σ или 1%/99% — стандартная практика).


In [ ]:
# === ВЫБРОСЫ — несколько методов ===
num_cols = df.select_dtypes(include=np.number).columns.tolist()

outlier_report = []
for col in num_cols:
    s = df[col].dropna()
    Q1, Q3 = s.quantile([0.25, 0.75])
    IQR = Q3 - Q1

    # IQR 1.5x и 3x
    n_iqr15 = ((s < Q1 - 1.5*IQR) | (s > Q3 + 1.5*IQR)).sum()
    n_iqr30 = ((s < Q1 - 3.0*IQR) | (s > Q3 + 3.0*IQR)).sum()

    # MAD-based (robust)
    med = s.median()
    mad = (s - med).abs().median()
    mod_z = 0.6745 * (s - med) / mad if mad > 0 else pd.Series(0, index=s.index)
    n_mad = (mod_z.abs() > 3.5).sum()

    outlier_report.append({
        "column":   col,
        "iqr_1.5x": n_iqr15,
        "iqr_3x":   n_iqr30,
        "mad":      n_mad,
        "pct_3x":   n_iqr30 / len(s) * 100,
    })

outlier_df = pd.DataFrame(outlier_report).sort_values("pct_3x", ascending=False)
print("Выбросы (разные методы):")
print(outlier_df.to_string(index=False))


### Target distribution — критичный шаг

Распределение таргета определяет:
- **Выбор функции потерь** (skew → MAE/Huber; bimodal → смешанная модель)
- **Нужна ли трансформация** (log-normal → `log(y)` + обратное преобразование)
- **Baseline** (для skewed regression baseline=median, не mean)
- **Метрику** (для асимметричного таргета — RMSE не всегда лучший выбор)

**Интерпретация skewness и kurtosis:**
- `skew ≈ 0` — симметричное; `|skew| > 1` — сильная асимметрия
- `kurt ≈ 0` (excess) — нормальное; `kurt > 3` — heavy tails (в финансах норма)
- Jarque-Bera test: `p < 0.05` → отклоняем нормальность


In [ ]:
# === РАСПРЕДЕЛЕНИЕ ТАРГЕТА ===
TARGET = "target"   # <-- ЗАМЕНИ НА РЕАЛЬНОЕ ИМЯ

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1. Гистограмма + KDE
axes[0].hist(df[TARGET].dropna(), bins=50, color="steelblue",
             edgecolor="white", density=True, alpha=0.7)
df[TARGET].dropna().plot(kind="kde", ax=axes[0], color="navy", linewidth=1.5)
axes[0].axvline(df[TARGET].mean(),   color="red",    linestyle="--", label=f"Mean={df[TARGET].mean():.2f}")
axes[0].axvline(df[TARGET].median(), color="orange", linestyle="--", label=f"Median={df[TARGET].median():.2f}")
axes[0].set_title(f"Распределение {TARGET}")
axes[0].legend()

# 2. Boxplot
axes[1].boxplot(df[TARGET].dropna(), vert=True)
axes[1].set_title(f"Boxplot {TARGET}")

# 3. Q-Q plot vs Normal
stats.probplot(df[TARGET].dropna(), dist="norm", plot=axes[2])
axes[2].set_title("Q-Q plot vs Normal")

plt.tight_layout()
plt.show()

# Сводка и тесты нормальности
print(df[TARGET].describe())
print(f"\nSkewness: {df[TARGET].skew():.3f}  (>1 или <-1 = сильный скос)")
print(f"Kurtosis: {df[TARGET].kurt():.3f}  (>3 = тяжёлые хвосты, <-1 = светлые)")

# Jarque-Bera test (H0: нормальное)
jb_stat, jb_p = jarque_bera(df[TARGET].dropna())
print(f"\nJarque-Bera: stat={jb_stat:.2f}, p={jb_p:.4g}")
print(f"  → Нормальность {'отвергается' if jb_p < 0.05 else 'не отвергается'}")


---
## 4. EDA — Univariate Analysis

Смотрим на каждую переменную отдельно.

- **Числовые:** форма распределения, скос, выбросы, нулевая дисперсия (константы)
- **Категориальные:** кардинальность, редкие категории (<1% от выборки)
- **Bool/binary:** баланс классов

**Вопрос:** *"Что делать с константными колонками?"* → Drop. Они бесполезны, раздувают матрицу, могут ломать VIF.

**Вопрос:** *"Что с high-cardinality (zip codes, user_id)?"*
→ Target encoding (с out-of-fold для защиты от leakage), frequency encoding, или эмбеддинги для нейросетей. OHE не масштабируется.


In [ ]:
# === ЧИСЛОВЫЕ ПЕРЕМЕННЫЕ ===
num_cols = df.select_dtypes(include=np.number).columns.tolist()
print(f"Числовых колонок: {len(num_cols)}")

# Константы — drop кандидаты
const_cols = [c for c in num_cols if df[c].nunique() <= 1]
if const_cols:
    print(f"⚠️  Константные колонки (drop): {const_cols}")

# Сетка гистограмм
plot_cols = [c for c in num_cols if c not in const_cols]
n_cols = 3
n_rows = (len(plot_cols) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 3))
axes = axes.flatten() if n_rows * n_cols > 1 else [axes]

for i, col in enumerate(plot_cols):
    axes[i].hist(df[col].dropna(), bins=40, color="steelblue", edgecolor="white", alpha=0.8)
    axes[i].set_title(f"{col}\nskew={df[col].skew():.2f}, kurt={df[col].kurt():.2f}", fontsize=9)
    axes[i].tick_params(labelsize=8)

for j in range(len(plot_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Распределения числовых переменных", y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# === КАТЕГОРИАЛЬНЫЕ ПЕРЕМЕННЫЕ ===
cat_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()
print(f"Категориальных колонок: {len(cat_cols)}")

cat_summary = []
for col in cat_cols:
    n_unique = df[col].nunique()
    top = df[col].value_counts(normalize=True).head(1)
    top_val, top_pct = (top.index[0], top.iloc[0]) if len(top) > 0 else (None, 0)

    # Редкие категории (<1%)
    vc = df[col].value_counts(normalize=True)
    rare = (vc < 0.01).sum()

    cat_summary.append({
        "column": col,
        "cardinality": n_unique,
        "top_value": top_val,
        "top_pct": f"{top_pct*100:.1f}%",
        "rare_(<1%)": rare,
    })

print(pd.DataFrame(cat_summary).to_string(index=False))

# Визуализация — только для low-cardinality
for col in cat_cols:
    n_unique = df[col].nunique()
    if n_unique <= 20:
        top_vals = df[col].value_counts().head(15)
        fig, ax = plt.subplots(figsize=(8, 3))
        top_vals.plot(kind="bar", ax=ax, color="steelblue")
        ax.set_title(f"{col} — частоты (top {len(top_vals)})")
        ax.tick_params(axis="x", rotation=45)
        plt.tight_layout()
        plt.show()
    else:
        print(f"{col}: {n_unique} уникальных — пропускаем визуализацию (используй target/frequency encoding)")


---
## 5. EDA — Bivariate Analysis

### Pearson vs Spearman — когда какой?

| Метрика | Что мерит | Когда предпочесть |
|---------|-----------|-------------------|
| **Pearson** | Линейную связь | Оба признака ~ normal, нет выбросов |
| **Spearman** | Монотонную связь (ранговую) | Heavy tails, выбросы, нелинейная монотонная связь |
| **Kendall** | То же что Spearman | Маленькая выборка |

**В квантах почти всегда используют Spearman** — цены/доходности имеют тяжёлые хвосты, Pearson сильно зависит от выбросов.

### Мультиколлинеарность

| Диагностика | Порог | Что делает |
|-------------|-------|------------|
| `|corr| > 0.8` | Первый сигнал | Смотрим пары |
| **VIF > 5** | Проблема | Обдумать дроп |
| **VIF > 10** | Серьёзная проблема | Дропнуть или регуляризовать |
| `cond(X) > 30` | Численная неустойчивость | QR / SVD / Ridge |

**VIF = 1 / (1 - R²_i)**, где R²_i — регрессия i-го признака на все остальные. Показывает, во сколько раз дисперсия веса раздута из-за коллинеарности.

**Вопросы:**
- *"Что если два признака имеют corr=0.95?"* → Один дропнуть (сохраняет интерпретируемость), либо Ridge/PCA (сохраняет информацию).
- *"Почему Lasso "выбирает" один из коррелированных?"* → Острый угол L1-шара → оптимум в углу, где зануляется один из весов. Выбор случайный; при слабо устойчивом отборе — ElasticNet.


In [ ]:
# === CORRELATION MATRIX ===
# Используем Spearman (robust к выбросам) + игнорируем константы
plot_num = [c for c in num_cols if df[c].nunique() > 1]
corr = df[plot_num].corr(method="spearman")

fig, ax = plt.subplots(figsize=(min(16, 1 + len(plot_num)*0.5), min(12, 1 + len(plot_num)*0.45)))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f",
            cmap="RdBu_r", center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.5, ax=ax,
            annot_kws={"size": 8})
ax.set_title("Correlation Matrix (Spearman)")
plt.tight_layout()
plt.show()

# Высоко коррелированные пары (мультиколлинеарность)
high_pairs = []
for i in range(len(corr)):
    for j in range(i+1, len(corr)):
        if abs(corr.iloc[i, j]) > 0.8:
            high_pairs.append((corr.index[i], corr.columns[j], corr.iloc[i, j]))

if high_pairs:
    print("\n⚠️  Высоко коррелированные пары (|ρ| > 0.8):")
    for a, b, r in sorted(high_pairs, key=lambda x: -abs(x[2])):
        print(f"   {a} ↔ {b}: ρ = {r:+.3f}")


In [ ]:
# === VIF (Variance Inflation Factor) ===
# Запускаем ТОЛЬКО на числовых non-const колонках без таргета
vif_cols = [c for c in plot_num if c != TARGET]
# Дропаем строки с NaN для корректного VIF
vif_df_data = df[vif_cols].dropna()

if len(vif_df_data) > len(vif_cols) + 10:  # нужна адекватная выборка
    # Стандартизация важна для численной устойчивости
    X_vif = StandardScaler().fit_transform(vif_df_data)

    vif_scores = pd.DataFrame({
        "feature": vif_cols,
        "VIF": [variance_inflation_factor(X_vif, i) for i in range(X_vif.shape[1])]
    }).sort_values("VIF", ascending=False)

    print("VIF scores:")
    print(vif_scores.to_string(index=False))
    print("\nПорог: VIF > 5 — подозрительно, VIF > 10 — серьёзная мультиколлинеарность")

    # Подсветка проблемных
    problem = vif_scores[vif_scores["VIF"] > 5]
    if len(problem) > 0:
        print(f"\n⚠️  Проблемных признаков: {len(problem)}")
else:
    print("Слишком мало данных для VIF")


In [ ]:
# === КОРРЕЛЯЦИЯ С ТАРГЕТОМ ===
# Сравниваем Pearson и Spearman — расхождение = нелинейность
feat_cols = [c for c in num_cols if c != TARGET and df[c].nunique() > 1]
target_corr = pd.DataFrame({
    "pearson":  [df[c].corr(df[TARGET], method="pearson")  for c in feat_cols],
    "spearman": [df[c].corr(df[TARGET], method="spearman") for c in feat_cols]
}, index=feat_cols)

target_corr["abs_spearman"] = target_corr["spearman"].abs()
target_corr["nonlinearity"] = (target_corr["spearman"] - target_corr["pearson"]).abs()
target_corr = target_corr.sort_values("abs_spearman", ascending=False)

print("Корреляция признаков с таргетом:")
print(target_corr.round(4).to_string())

# График
fig, ax = plt.subplots(figsize=(10, max(4, len(feat_cols)*0.3)))
target_corr[["pearson", "spearman"]].plot(kind="barh", ax=ax, width=0.8)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title(f"Pearson vs Spearman корреляция с {TARGET}")
ax.legend()
plt.tight_layout()
plt.show()

# Признаки с сильной нелинейной связью
nonlinear = target_corr[target_corr["nonlinearity"] > 0.1]
if len(nonlinear) > 0:
    print(f"\n⚠️  Возможная нелинейность (|Spearman - Pearson| > 0.1):")
    print(nonlinear[["pearson", "spearman", "nonlinearity"]].to_string())


In [ ]:
# === SCATTER PLOTS топ признаков с таргетом ===
top_features = target_corr.head(6).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(top_features):
    mask_valid = df[[col, TARGET]].dropna()
    axes[i].scatter(mask_valid[col], mask_valid[TARGET],
                    alpha=0.3, s=10, color="steelblue")

    # Линия тренда (linear fit)
    z = np.polyfit(mask_valid[col], mask_valid[TARGET], 1)
    x_line = np.linspace(mask_valid[col].min(), mask_valid[col].max(), 100)
    axes[i].plot(x_line, np.poly1d(z)(x_line), "r--", linewidth=1.5, label="linear")

    # LOWESS для нелинейной формы (если данных мало, subsample)
    if len(mask_valid) <= 2000:
        lowess = sm.nonparametric.lowess(mask_valid[TARGET], mask_valid[col], frac=0.3)
        axes[i].plot(lowess[:, 0], lowess[:, 1], "g-", linewidth=1.5, label="lowess")
    axes[i].legend(fontsize=7)

    r_p = df[col].corr(df[TARGET], method="pearson")
    r_s = df[col].corr(df[TARGET], method="spearman")
    axes[i].set_title(f"{col}\nPearson={r_p:.3f} | Spearman={r_s:.3f}", fontsize=9)
    axes[i].set_xlabel(col, fontsize=8)
    axes[i].set_ylabel(TARGET, fontsize=8)

plt.suptitle("Топ признаки vs таргет (linear vs LOWESS)", y=1.01)
plt.tight_layout()
plt.show()


---
## 6. Feature Engineering

> **Правило железное:** все трансформации с *learned statistics* (mean, std, quantiles, ranks) — только на train, потом применяем к test. Иначе **data leakage**.

### Типичные трансформации

| Трансформация | Когда | Формула |
|--------------|-------|---------|
| **log1p** | `skew > 1`, `x ≥ 0` | `log(1+x)` |
| **sqrt** | Умеренный правый скос, `x ≥ 0` | `√x` |
| **Yeo-Johnson** | Универсально (работает с отрицательными) | Обобщение Box-Cox |
| **Box-Cox** | `x > 0`, хотим строго нормальное | Требует положительности |
| **Ранговая норм.** | Quant factors, robust to outliers | `(rank - 0.5) / n - 0.5` |
| **Winsorization** | Клиппинг выбросов без дропа | `clip(x, q_low, q_high)` |

### Почему cross-sectional ranks в квантах?

В факторных моделях сигнал *относительный*: не абсолютное значение P/E, а "дёшев ли актив по сравнению с другими *в этот же день*". Поэтому ранги считаются **внутри каждой даты**, не глобально.

```python
df["rank_factor"] = df.groupby("date")["factor"].transform(
    lambda x: (x.rank() - 1) / (x.count() - 1) - 0.5
)
```

### Взаимодействия

Линейная модель не видит `x1 * x2` без явной фичи. Для GBM — менее критично (деревья сами находят), но явные взаимодействия по доменным гипотезам всегда помогают.

**Вопрос:** *"Feature engineering vs model complexity — что важнее?"*
→ В табличных задачах FE доминирует. GBM на хороших фичах бьёт нейронку на сырых данных почти всегда.


In [ ]:
# === ВАЖНО: FE ДЕЛАЕМ ПОСЛЕ SPLIT ===
# Все статистики (skew, quantiles, means) считаем на train,
# применяем к test тем же трансформером.

# Сначала определяем taргет и разделяем
TARGET = "target"  # проверь имя
FEATURES_RAW = [c for c in df.columns if c != TARGET]

# Убираем константы (были найдены выше)
FEATURES_RAW = [c for c in FEATURES_RAW if df[c].nunique() > 1]

X = df[FEATURES_RAW].copy()
y = df[TARGET].copy()

# --- Сплит (случайный, для iid данных) ---
# Для temporal — использовать split по времени (см. альтернативу ниже)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED
)

# --- Альтернатива: temporal split ---
# df_sorted = df.sort_values("date").reset_index(drop=True)
# split_idx = int(len(df_sorted) * 0.8)
# X_train = df_sorted[FEATURES_RAW].iloc[:split_idx]
# X_test  = df_sorted[FEATURES_RAW].iloc[split_idx:]
# y_train = df_sorted[TARGET].iloc[:split_idx]
# y_test  = df_sorted[TARGET].iloc[split_idx:]

print(f"Train: {X_train.shape} | Test: {X_test.shape}")


In [ ]:
# === WINSORIZATION (опционально, для quant factors) ===
# Клиппируем экстремумы, вычисляя границы ТОЛЬКО на train
def winsorize_fit(train_series, lower_q=0.01, upper_q=0.99):
    """Возвращает функцию-трансформер, фитнутую на train."""
    lo = train_series.quantile(lower_q)
    hi = train_series.quantile(upper_q)
    return lambda s: s.clip(lo, hi), (lo, hi)

# Применяем к сильно скошенным признакам
numeric_train_cols = X_train.select_dtypes(include=np.number).columns.tolist()
skewed_to_winsor = [c for c in numeric_train_cols if abs(X_train[c].skew()) > 3]

print(f"Winsorize (|skew| > 3): {skewed_to_winsor}")
winsor_bounds = {}
for col in skewed_to_winsor:
    fn, bounds = winsorize_fit(X_train[col])
    X_train[col] = fn(X_train[col])
    X_test[col]  = X_test[col].clip(*bounds)  # ВАЖНО: границы от train
    winsor_bounds[col] = bounds

if winsor_bounds:
    print("Winsorization bounds (из train):")
    for c, (lo, hi) in winsor_bounds.items():
        print(f"   {c}: [{lo:.3f}, {hi:.3f}]")


In [ ]:
# === YEO-JOHNSON для skewed признаков ===
# Универсальная трансформация — работает с отрицательными и с нулями.
# Это более правильный выбор, чем log1p, особенно для доходностей.

numeric_train_cols = X_train.select_dtypes(include=np.number).columns.tolist()
skewed_to_transform = [c for c in numeric_train_cols
                        if abs(X_train[c].skew()) > 1]

print(f"Yeo-Johnson для: {skewed_to_transform}")

if skewed_to_transform:
    pt = PowerTransformer(method="yeo-johnson", standardize=False)
    # fit ТОЛЬКО на train
    X_train[skewed_to_transform] = pt.fit_transform(X_train[skewed_to_transform])
    # transform на test — тем же fitted-объектом
    X_test[skewed_to_transform] = pt.transform(X_test[skewed_to_transform])

    print("Лямбды Yeo-Johnson (близко к 1 — почти identity, <0 — инверсия):")
    for col, lam in zip(skewed_to_transform, pt.lambdas_):
        print(f"   {col}: λ = {lam:+.3f}")


In [ ]:
# === ВЗАИМОДЕЙСТВИЯ (доменные) ===
# Добавляем явные произведения/отношения признаков.
# Примеры ниже — заглушки, адаптируй под свой датасет.

# X_train["feat1_x_feat2"] = X_train["feat1"] * X_train["feat2"]
# X_test ["feat1_x_feat2"] = X_test ["feat1"] * X_test ["feat2"]

# Безопасное отношение (против деления на 0)
# X_train["feat1_ratio_feat2"] = X_train["feat1"] / (X_train["feat2"].abs() + 1e-8)
# X_test ["feat1_ratio_feat2"] = X_test ["feat1"] / (X_test ["feat2"].abs() + 1e-8)

print(f"Shape после FE: train {X_train.shape}, test {X_test.shape}")


In [ ]:
# === CROSS-SECTIONAL RANKS (для quant datasets с временной структурой) ===
# Используй только если есть колонка даты и нужен факторный подход.
# Ранги считаются ВНУТРИ каждой даты — не глобально.

# date_col = "date"
# factor_cols = ["pe", "pb", "momentum_12m"]
#
# for col in factor_cols:
#     # Важно: groupby.transform не вызывает leakage, если делается внутри каждого периода
#     X_train[f"rank_{col}"] = (
#         X_train.assign(_date=df.loc[X_train.index, date_col])
#                .groupby("_date")[col]
#                .transform(lambda x: (x.rank() - 1) / (x.count() - 1) - 0.5)
#     )
#     X_test[f"rank_{col}"] = (
#         X_test.assign(_date=df.loc[X_test.index, date_col])
#               .groupby("_date")[col]
#               .transform(lambda x: (x.rank() - 1) / (x.count() - 1) - 0.5)
#     )
print("Cross-sectional ranks: пример выше, закомментирован")


---
## 7. Preprocessing & ColumnTransformer Pipeline

**Золотое правило:** test никогда не участвует в `fit()`. Pipeline это гарантирует автоматически.

### Почему обязательно масштабировать для линейных моделей?

1. **Без масштабирования L1/L2 штрафуют неравномерно** — признак в миллионах получит маленький вес (чтобы не раздуть штраф), признак в единицах — большой. Это не про важность, это про шкалу.
2. **GD сходится медленно** на плохо обусловленной задаче (вытянутые эллипсы уровня).
3. **Интерпретация коэффициентов** после стандартизации: коэфф. = эффект на y при изменении x на 1σ.

### Scaler cheatsheet

| Scaler | Формула | Когда |
|--------|---------|-------|
| `StandardScaler` | `(x - μ) / σ` | Нормально распределённые признаки |
| `RobustScaler` | `(x - median) / IQR` | Выбросы; факторы в квантах |
| `MinMaxScaler` | `(x - min) / (max - min)` | Когда нужен диапазон [0,1] (нейросети) |
| `PowerTransformer` | Yeo-Johnson / Box-Cox | Сильно скошенные |

### Категориальные: OneHot vs Target Encoding

| Подход | Когда |
|--------|-------|
| **OneHot (`drop='if_binary'`)** | Cardinality < ~30. Для линейных моделей обязательно drop, иначе dummy trap |
| **Ordinal** | Есть естественный порядок (small < medium < large) |
| **Target encoding (с K-fold out-of-fold)** | High cardinality (user_id, zip) |
| **Frequency encoding** | High cardinality + не нужна связь с таргетом |

**Вопрос-ловушка:** *"Дамми-трэп в OneHot — что это и когда важно?"*
→ Сумма всех OHE-столбцов = единичный вектор = свободный член. Матрица вырожденная. Линейной регрессии без регуляризации — больно. Ridge/Lasso терпят, но теряется интерпретируемость.


In [ ]:
# === PREPROCESSING PIPELINE ===
# Определяем колонки по типам (внутри X_train, после FE)
num_features = X_train.select_dtypes(include=np.number).columns.tolist()
cat_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

print(f"Numeric features:     {len(num_features)}")
print(f"Categorical features: {len(cat_features)}")
if cat_features:
    print(f"   {cat_features}")

# Пайплайны для каждого типа
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler()),
])

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore",
                              drop="if_binary",       # избегаем dummy trap
                              sparse_output=False)),
])

# ColumnTransformer склеивает
preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_features),
    ("cat", cat_pipeline, cat_features),
], remainder="drop", verbose_feature_names_out=False)

# Fit ТОЛЬКО на train
X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc  = preprocessor.transform(X_test)     # transform, не fit_transform!

# Сохраняем feature names для интерпретации
feature_names = preprocessor.get_feature_names_out()
print(f"\nПосле preprocessing: {X_train_proc.shape[1]} фичей")
print(f"Первые 10: {list(feature_names[:10])}")


---
## 8. Baseline Model

**Золотое правило:** сложная модель должна *заметно* бить baseline. Если нет — либо данные не содержат сигнала, либо пайплайн сломан.

### Выбор baseline

| Задача | Baseline |
|--------|----------|
| Регрессия | `DummyRegressor(strategy="median")` для skewed, `"mean"` для симметричных |
| Бинарная классификация | `DummyClassifier(strategy="stratified")` или `"most_frequent"` |
| Временной ряд | Last value (`y_pred = y[t-1]`), moving average |
| Факторная модель | Sharpe = 0 (no skill) + equal-weight бенчмарк |

**Вопрос:** *"Почему DummyRegressor даёт R² = 0 на train, а на test может быть < 0?"*
→ R² = 1 - SS_res/SS_tot. Dummy даёт SS_res = SS_tot на train (идеальное предсказание среднего), поэтому R² = 0. На test среднее из train ≠ среднее из test → R² < 0.


In [ ]:
# === UNIVERSAL EVALUATOR ===
def evaluate_regression(model, X_tr, y_tr, X_te, y_te, name="Model"):
    """Обучает и оценивает регрессионную модель."""
    model.fit(X_tr, y_tr)
    y_pred_tr = model.predict(X_tr)
    y_pred_te = model.predict(X_te)

    metrics = {
        "model":      name,
        "RMSE_train": np.sqrt(mean_squared_error(y_tr, y_pred_tr)),
        "RMSE_test":  np.sqrt(mean_squared_error(y_te, y_pred_te)),
        "MAE_test":   mean_absolute_error(y_te, y_pred_te),
        "R2_train":   r2_score(y_tr, y_pred_tr),
        "R2_test":    r2_score(y_te, y_pred_te),
    }
    metrics["overfit_gap"] = metrics["RMSE_test"] - metrics["RMSE_train"]

    print(f"\n{'='*55}")
    print(f"  {name}")
    print(f"{'='*55}")
    print(f"  RMSE train: {metrics['RMSE_train']:.4f}")
    print(f"  RMSE test:  {metrics['RMSE_test']:.4f}  (gap: {metrics['overfit_gap']:+.4f})")
    print(f"  MAE  test:  {metrics['MAE_test']:.4f}")
    print(f"  R²   train: {metrics['R2_train']:.4f}")
    print(f"  R²   test:  {metrics['R2_test']:.4f}")

    return metrics


# --- Baseline ---
baseline_mean   = DummyRegressor(strategy="mean")
baseline_median = DummyRegressor(strategy="median")

results = []
results.append(evaluate_regression(baseline_mean,   X_train_proc, y_train, X_test_proc, y_test, "Baseline (mean)"))
results.append(evaluate_regression(baseline_median, X_train_proc, y_train, X_test_proc, y_test, "Baseline (median)"))


---
## 9. ML Models

### Порядок усложнения

1. **Linear + RidgeCV** — интерпретируемый baseline, автотюнинг α
2. **Lasso / ElasticNet** — feature selection, если много шумовых фичей
3. **Huber / Quantile** — robust альтернативы при выбросах
4. **LightGBM / XGBoost** — капитуляция на нелинейности

### Ridge vs Lasso vs ElasticNet — когда какой

| Модель | Регуляризация | Эффект | Когда |
|--------|---------------|--------|-------|
| **Ridge** | `Σw²` | Равномерно сжимает веса | Все фичи немного полезны; мультиколлинеарность |
| **Lasso** | `Σ|w|` | Зануляет часть весов | Много шумовых признаков; нужен отбор |
| **ElasticNet** | `α·Σw² + (1-α)·Σ|w|` | Гибрид | Скоррелированные группы признаков (Lasso случайно выбирает одного, ElasticNet — группу) |

### LightGBM vs XGBoost — в чём разница

| Аспект | LightGBM | XGBoost |
|--------|----------|---------|
| Рост деревьев | **Leaf-wise** (в глубину важного листа) | **Level-wise** (все листья одного уровня) |
| Скорость | Быстрее на больших данных | Чуть медленнее |
| Overfitting | Легче переобучить (leaf-wise) | Консервативнее |
| Категории | **Нативная** поддержка | Нужен OHE/target encoding |
| Регуляризация | `lambda_l1/l2`, `min_child_samples` | `reg_alpha/lambda`, `max_depth` |

**Вопрос:** *"Почему бустинг обычно бьёт случайный лес?"*
→ RF усредняет независимые деревья (variance reduction, bias не уменьшается). Бустинг добавляет деревья последовательно, каждое исправляет ошибки предыдущих → уменьшает и bias, и variance.

**Вопрос:** *"Что делать если бустинг переобучился?"*
→ `learning_rate↓` + `n_estimators↑`, `min_child_samples↑`, `reg_lambda↑`, `max_depth↓`, раннюю остановку на validation.


In [ ]:
# === RIDGE CV — автоподбор alpha ===
# Почему RidgeCV а не просто Ridge(alpha=1.0):
# - интервьюер обязательно спросит "как выбрал alpha?";
# - встроенный leave-one-out CV по умолчанию — дёшево, быстро.

alphas = np.logspace(-3, 3, 30)
ridge = RidgeCV(alphas=alphas, cv=5, scoring="neg_root_mean_squared_error")
results.append(evaluate_regression(ridge, X_train_proc, y_train, X_test_proc, y_test, "RidgeCV"))
print(f"\nBest alpha: {ridge.alpha_:.4f}")


In [ ]:
# === LASSO CV ===
lasso = LassoCV(alphas=alphas, cv=5, max_iter=10000, random_state=SEED)
results.append(evaluate_regression(lasso, X_train_proc, y_train, X_test_proc, y_test, "LassoCV"))
print(f"Best alpha: {lasso.alpha_:.4f}")
print(f"Занулённых признаков: {(lasso.coef_ == 0).sum()} из {len(lasso.coef_)}")


In [ ]:
# === ELASTIC NET CV ===
enet = ElasticNetCV(
    alphas=alphas,
    l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9],
    cv=5, max_iter=10000, random_state=SEED
)
results.append(evaluate_regression(enet, X_train_proc, y_train, X_test_proc, y_test, "ElasticNetCV"))
print(f"Best alpha: {enet.alpha_:.4f}, l1_ratio: {enet.l1_ratio_:.2f}")


In [ ]:
# === RIDGE COEFFICIENTS (интерпретация) ===
coef_df = pd.DataFrame({
    "feature": feature_names,
    "coef":    ridge.coef_
}).sort_values("coef", key=abs, ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ["steelblue" if c > 0 else "coral" for c in coef_df["coef"]]
ax.barh(coef_df["feature"], coef_df["coef"], color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title(f"Ridge — топ-20 коэффициентов (α={ridge.alpha_:.3f})")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

# Интерпретация: после StandardScaler коэфф = изменение y при сдвиге x на 1σ
print("\nТоп-5 признаков:")
print(coef_df.head().to_string(index=False))


In [ ]:
# === LIGHTGBM ===
# ВАЖНО: subsample требует subsample_freq >= 1, иначе игнорируется!
lgb_model = lgb.LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=63,
    max_depth=-1,              # без ограничения, контролируем через num_leaves
    min_child_samples=20,
    subsample=0.8,
    subsample_freq=1,          # ← обязательно для реального bagging
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=SEED,
    verbose=-1,
)

# Early stopping через callbacks (для temporal — нужен временной split внутри train)
# Здесь используем простой validation split из train
from sklearn.model_selection import train_test_split as tts_val
X_tr_in, X_val_in, y_tr_in, y_val_in = tts_val(
    X_train_proc, y_train, test_size=0.2, random_state=SEED
)

lgb_model.fit(
    X_tr_in, y_tr_in,
    eval_set=[(X_val_in, y_val_in)],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)]
)

results.append({
    "model": "LightGBM",
    "RMSE_train": np.sqrt(mean_squared_error(y_train, lgb_model.predict(X_train_proc))),
    "RMSE_test":  np.sqrt(mean_squared_error(y_test,  lgb_model.predict(X_test_proc))),
    "MAE_test":   mean_absolute_error(y_test, lgb_model.predict(X_test_proc)),
    "R2_train":   r2_score(y_train, lgb_model.predict(X_train_proc)),
    "R2_test":    r2_score(y_test,  lgb_model.predict(X_test_proc)),
})
results[-1]["overfit_gap"] = results[-1]["RMSE_test"] - results[-1]["RMSE_train"]
print(f"\nLightGBM: RMSE_test={results[-1]['RMSE_test']:.4f}, best_iter={lgb_model.best_iteration_}")


In [ ]:
# === XGBOOST ===
xgb_model = xgb.XGBRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=SEED,
    verbosity=0,
    early_stopping_rounds=50,
)

xgb_model.fit(X_tr_in, y_tr_in, eval_set=[(X_val_in, y_val_in)], verbose=False)

results.append({
    "model": "XGBoost",
    "RMSE_train": np.sqrt(mean_squared_error(y_train, xgb_model.predict(X_train_proc))),
    "RMSE_test":  np.sqrt(mean_squared_error(y_test,  xgb_model.predict(X_test_proc))),
    "MAE_test":   mean_absolute_error(y_test, xgb_model.predict(X_test_proc)),
    "R2_train":   r2_score(y_train, xgb_model.predict(X_train_proc)),
    "R2_test":    r2_score(y_test,  xgb_model.predict(X_test_proc)),
})
results[-1]["overfit_gap"] = results[-1]["RMSE_test"] - results[-1]["RMSE_train"]
print(f"\nXGBoost: RMSE_test={results[-1]['RMSE_test']:.4f}, best_iter={xgb_model.best_iteration}")


In [ ]:
# === СРАВНИТЕЛЬНАЯ ТАБЛИЦА ===
results_df = pd.DataFrame(results).set_index("model")
print("\nСравнение моделей:")
print(results_df.round(4).to_string())

# График train vs test
fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(results_df))
width = 0.35
ax.bar(x - width/2, results_df["RMSE_train"], width, label="Train", color="steelblue", alpha=0.8)
ax.bar(x + width/2, results_df["RMSE_test"],  width, label="Test",  color="coral",     alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(results_df.index, rotation=20)
ax.set_ylabel("RMSE")
ax.set_title("Train vs Test RMSE")
ax.legend()
plt.tight_layout()
plt.show()

# Кандидат overfitting: большой gap
worst_overfit = results_df["overfit_gap"].idxmax()
print(f"\nНаибольший overfit: {worst_overfit} (gap={results_df.loc[worst_overfit, 'overfit_gap']:+.4f})")


In [ ]:
# === FEATURE IMPORTANCE — LightGBM (GAIN, не split!) ===
# По умолчанию feature_importances_ = split count.
# Gain = средний вклад признака в снижение loss — предпочтительная метрика.

gain_importance = lgb_model.booster_.feature_importance(importance_type="gain")
split_importance = lgb_model.booster_.feature_importance(importance_type="split")

fi = pd.DataFrame({
    "feature": feature_names,
    "gain":    gain_importance,
    "split":   split_importance,
})
fi["gain_pct"] = fi["gain"] / fi["gain"].sum() * 100
fi = fi.sort_values("gain", ascending=False).head(20)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].barh(fi["feature"], fi["gain_pct"], color="steelblue")
axes[0].set_title("LightGBM — Gain importance (%)")
axes[0].invert_yaxis()

axes[1].barh(fi["feature"], fi["split"], color="coral")
axes[1].set_title("LightGBM — Split count")
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

print("\nТоп-10 по gain:")
print(fi[["feature", "gain", "gain_pct"]].head(10).to_string(index=False))


In [ ]:
# === PERMUTATION IMPORTANCE (model-agnostic, более надёжная) ===
# Gain может вводить в заблуждение (биас к категориалкам high-cardinality).
# Permutation importance — перемешиваем колонку, смотрим насколько упала метрика.
from sklearn.inspection import permutation_importance

perm = permutation_importance(
    lgb_model, X_test_proc, y_test,
    n_repeats=10, random_state=SEED, n_jobs=-1, scoring="neg_root_mean_squared_error"
)

perm_df = pd.DataFrame({
    "feature": feature_names,
    "importance_mean": perm.importances_mean,
    "importance_std":  perm.importances_std,
}).sort_values("importance_mean", ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(perm_df["feature"], perm_df["importance_mean"],
        xerr=perm_df["importance_std"], color="steelblue", alpha=0.8, capsize=3)
ax.invert_yaxis()
ax.set_title("Permutation Importance (на test, LightGBM)")
ax.set_xlabel("Падение RMSE при перемешивании (выше = важнее)")
plt.tight_layout()
plt.show()


---
## 10. Validation — для разных типов данных

### Почему k-fold не работает для временных рядов

В обычном k-fold train может содержать точки *после* test. Для временных данных это **lookahead bias**: модель "видит будущее" во время обучения. Метрики на такой CV завышены, а на продакшене — разочарование.

### Walk-forward (TimeSeriesSplit)

```
Fold 1:  [train train train]  [test]
Fold 2:  [train train train train]  [test]
Fold 3:  [train train train train train]  [test]
```
Это expanding window. Есть и sliding window (фиксированный train).

### Purged CV with Embargo (López de Prado)

Если у таргета есть **overlap** (например, forward returns 5 дней — `y_t` зависит от `t` до `t+5`), простой walk-forward имеет leak: точка `train[t-2]` знает про `return_{t-2:t+3}`, а `test[t]` про `return_{t:t+5}` — они **пересекаются**.

**Решение:**
1. **Purge** — убираем из train точки, чьи таргеты пересекаются с test
2. **Embargo** — добавляем "буфер" после test, чтобы следующий train не брал точки слишком близкие

Это must-have для реалистичной quant validation.

### Stratified K-fold
Только для **классификации**: сохраняет пропорции классов в каждом fold. Для несбалансированных задач обязателен.


In [ ]:
# === WALK-FORWARD VALIDATION ===
def walk_forward_validation(model_factory, X, y, n_splits=5, verbose=True):
    """
    Expanding window walk-forward CV.

    model_factory — callable, возвращающий свежий экземпляр модели.
                    Важно: переиспользовать обученную модель нельзя (leakage через state).
    """
    tscv = TimeSeriesSplit(n_splits=n_splits)
    fold_metrics = []

    for fold, (train_idx, test_idx) in enumerate(tscv.split(X)):
        X_tr, X_te = X[train_idx], X[test_idx]
        y_tr = y.iloc[train_idx] if hasattr(y, "iloc") else y[train_idx]
        y_te = y.iloc[test_idx]  if hasattr(y, "iloc") else y[test_idx]

        m = model_factory()
        m.fit(X_tr, y_tr)
        y_pred = m.predict(X_te)

        rmse = np.sqrt(mean_squared_error(y_te, y_pred))
        mae  = mean_absolute_error(y_te, y_pred)
        r2   = r2_score(y_te, y_pred)
        fold_metrics.append({
            "fold": fold+1, "n_train": len(train_idx), "n_test": len(test_idx),
            "RMSE": rmse, "MAE": mae, "R2": r2
        })
        if verbose:
            print(f"  Fold {fold+1}: train={len(train_idx):5d}, test={len(test_idx):4d} "
                  f"| RMSE={rmse:.4f}, R²={r2:+.4f}")

    metrics_df = pd.DataFrame(fold_metrics)
    print(f"\n  Mean RMSE: {metrics_df['RMSE'].mean():.4f} ± {metrics_df['RMSE'].std():.4f}")
    print(f"  Mean R²:   {metrics_df['R2'].mean():+.4f} ± {metrics_df['R2'].std():.4f}")
    return metrics_df

# Для temporal данных: сначала сортировка по дате, потом WFV на ВСЕХ данных
# df_sorted = df.sort_values("date").reset_index(drop=True)
# X_all_proc = preprocessor.transform(df_sorted[FEATURES_RAW])
# y_all = df_sorted[TARGET]
# wf_results = walk_forward_validation(lambda: Ridge(alpha=1.0), X_all_proc, y_all)

print("Walk-Forward — Ridge (on train set for illustration):")
wf_results = walk_forward_validation(
    lambda: RidgeCV(alphas=alphas),
    X_train_proc, y_train, n_splits=5
)


In [ ]:
# === PURGED CV WITH EMBARGO (López de Prado) ===
class PurgedKFold:
    """
    K-Fold с purge (удаление из train пересекающихся с test точек)
    и embargo (буфер после test).

    Использование: когда y_t зависит от данных [t, t+h] (forward returns).
    """
    def __init__(self, n_splits=5, embargo_pct=0.01, purge_size=0):
        self.n_splits = n_splits
        self.embargo_pct = embargo_pct  # доля выборки в буфер
        self.purge_size  = purge_size   # horizon таргета (на сколько периодов он "смотрит вперёд")

    def split(self, X):
        n = len(X)
        indices = np.arange(n)
        test_size = n // self.n_splits
        embargo = int(n * self.embargo_pct)

        for fold in range(self.n_splits):
            test_start = fold * test_size
            test_end   = test_start + test_size
            test_idx   = indices[test_start:test_end]

            # Train = всё, кроме test + purge (до test) + embargo (после test)
            purge_start = max(0, test_start - self.purge_size)
            embargo_end = min(n, test_end + embargo)

            train_idx = np.concatenate([
                indices[:purge_start],
                indices[embargo_end:]
            ])
            yield train_idx, test_idx


# Пример: forward 5-day return — horizon = 5, embargo = 1%
print("Purged K-Fold example (horizon=5, embargo=1%):")
pcv = PurgedKFold(n_splits=5, embargo_pct=0.01, purge_size=5)
for fold, (tr, te) in enumerate(pcv.split(X_train_proc)):
    print(f"  Fold {fold+1}: train={len(tr)} (purged+embargoed), test={len(te)}")


In [ ]:
# === LEARNING CURVES (bias-variance диагностика) ===
# Кривые показывают:
#  - train и val сходятся высоко → high bias (underfit) → нужна более сложная модель
#  - train низко, val высоко (большой gap) → high variance (overfit) → регуляризация / больше данных

train_sizes, train_scores, val_scores = learning_curve(
    RidgeCV(alphas=alphas),
    X_train_proc, y_train,
    train_sizes=np.linspace(0.1, 1.0, 10),
    cv=5, scoring="neg_root_mean_squared_error",
    n_jobs=-1, random_state=SEED
)

train_rmse = -train_scores.mean(axis=1)
val_rmse   = -val_scores.mean(axis=1)
train_std  = train_scores.std(axis=1)
val_std    = val_scores.std(axis=1)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(train_sizes, train_rmse, "o-", color="steelblue", label="Train")
ax.fill_between(train_sizes, train_rmse - train_std, train_rmse + train_std,
                alpha=0.2, color="steelblue")
ax.plot(train_sizes, val_rmse, "o-", color="coral", label="Validation")
ax.fill_between(train_sizes, val_rmse - val_std, val_rmse + val_std,
                alpha=0.2, color="coral")
ax.set_xlabel("Размер train")
ax.set_ylabel("RMSE")
ax.set_title("Learning Curves — Ridge")
ax.legend()
plt.tight_layout()
plt.show()

gap = val_rmse[-1] - train_rmse[-1]
print(f"\nFinal gap (val - train RMSE): {gap:+.4f}")
if gap > 0.1 * train_rmse[-1]:
    print("  → Возможен overfit: либо больше данных, либо сильнее регуляризация")
elif val_rmse[-1] > 2 * train_rmse[0]:
    print("  → Возможен underfit: модель слишком простая")
else:
    print("  → Баланс выглядит разумным")


---
## 11. Quant Analysis — IC / ICIR / Quintiles

Этот блок специфичен для quant research (предсказание доходностей / ранжирование активов).

### Ключевые метрики

**IC (Information Coefficient)** — Spearman(predicted, realized returns), считается **на каждую дату** (cross-sectionally).
- `IC ≈ 0.02–0.05` — рабочий фактор
- `IC > 0.1` — либо сильный сигнал, либо leak (проверь!)

**ICIR** — аналог Sharpe для факторов: `mean(IC) / std(IC)`.
- `ICIR > 0.5` — хорошо
- `ICIR > 1.0` — очень хорошо

**t-statistic для ICIR:** `t = ICIR × √N`, где N — количество периодов.
- `|t| > 2` — значимо на 5%
- **Но:** IC имеют автокорреляцию → простая t-stat завышает значимость. Используй **Newey-West** HAC errors.

### Квинтильный анализ

Ранжируем активы по фактору в каждый период, делим на 5 групп (Q1 = низкие, Q5 = высокие), смотрим доходности:
- **Spread Q5 - Q1** — потенциальная прибыль long-short портфеля
- **Монотонность** — если средние доходности не монотонны от Q1 к Q5, фактор шумный
- **Turnover** — сколько активов меняют квинтиль между периодами (высокий → transaction costs съедят alpha)

**Вопрос:** *"У меня IC = 0.03 и ICIR = 0.8. Это хороший фактор?"*
→ Зависит от capacity, turnover, корреляции с существующими. В Renaissance IC = 0.01 достаточно, в маленьком фонде — мало. Всегда смотреть в контексте.


In [ ]:
# === IC / ICIR — правильная реализация с Newey-West ===

def compute_ic_series(df_quant, factor_col, return_col, date_col):
    """
    Считает cross-sectional Spearman IC на каждую дату.

    Args:
        df_quant  — DataFrame с фактором, доходностью, датой
        factor_col, return_col, date_col — имена колонок
    Returns:
        pd.Series, индекс — даты, значения — IC
    """
    ic_series = (
        df_quant.dropna(subset=[factor_col, return_col])
                .groupby(date_col)
                .apply(lambda g: spearmanr(g[factor_col], g[return_col],
                                           nan_policy="omit")[0]
                       if len(g) >= 3 else np.nan)
                .dropna()
    )
    return ic_series


def compute_icir(ic_series, nw_lags=5):
    """
    ICIR + t-stat (как наивный, так и Newey-West adjusted).

    Newey-West корректирует t-stat на автокорреляцию в IC.
    Для дневных данных lags=5, для месячных lags=3-6.
    """
    n = len(ic_series)
    mean_ic = ic_series.mean()
    std_ic  = ic_series.std(ddof=1)
    icir    = mean_ic / std_ic if std_ic > 0 else 0

    # Наивный t-stat (предполагает iid)
    t_naive = icir * np.sqrt(n)
    p_naive = 2 * (1 - stats.t.cdf(abs(t_naive), df=n-1))

    # Newey-West t-stat (HAC errors)
    X = np.ones((n, 1))
    try:
        model = sm.OLS(ic_series.values, X).fit(
            cov_type="HAC", cov_kwds={"maxlags": nw_lags}
        )
        t_nw = model.tvalues[0]
        p_nw = model.pvalues[0]
    except Exception as e:
        t_nw, p_nw = np.nan, np.nan

    return {
        "mean_IC":      mean_ic,
        "std_IC":       std_ic,
        "ICIR":         icir,
        "t_naive":      t_naive,
        "p_naive":      p_naive,
        "t_NeweyWest":  t_nw,
        "p_NeweyWest":  p_nw,
        "n_periods":    n,
        "IC>0_pct":     (ic_series > 0).mean() * 100,  # hit rate
    }


def plot_ic_analysis(ic_series, title="IC Analysis", nw_lags=5):
    stats_dict = compute_icir(ic_series, nw_lags=nw_lags)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    # 1. IC по времени
    ic_series.plot(ax=axes[0], color="steelblue", linewidth=1)
    axes[0].axhline(0, color="black", linewidth=0.8)
    axes[0].axhline(ic_series.mean(), color="red", linestyle="--",
                    label=f"Mean={ic_series.mean():.4f}")
    axes[0].fill_between(range(len(ic_series)), ic_series, 0,
                         where=ic_series > 0, color="steelblue", alpha=0.3)
    axes[0].fill_between(range(len(ic_series)), ic_series, 0,
                         where=ic_series < 0, color="coral", alpha=0.3)
    axes[0].set_title("IC по периодам")
    axes[0].legend(fontsize=9)

    # 2. Гистограмма IC
    axes[1].hist(ic_series, bins=25, color="steelblue", edgecolor="white", alpha=0.8)
    axes[1].axvline(0, color="black", linewidth=0.8)
    axes[1].axvline(ic_series.mean(), color="red", linestyle="--")
    axes[1].set_title("Распределение IC")
    axes[1].set_xlabel("IC")

    # 3. Кумулятивный IC
    ic_series.cumsum().reset_index(drop=True).plot(
        ax=axes[2], color="steelblue", linewidth=1.5
    )
    axes[2].axhline(0, color="black", linewidth=0.8)
    axes[2].set_title("Кумулятивный IC")

    plt.suptitle(
        f"{title} | Mean IC={stats_dict['mean_IC']:.4f} | "
        f"ICIR={stats_dict['ICIR']:.3f} | "
        f"t_NW={stats_dict['t_NeweyWest']:.2f} (p={stats_dict['p_NeweyWest']:.4f}) | "
        f"hit rate={stats_dict['IC>0_pct']:.1f}%"
    )
    plt.tight_layout()
    plt.show()

    print("\nIC Statistics:")
    for k, v in stats_dict.items():
        print(f"  {k:15s}: {v:.4f}" if isinstance(v, (int, float)) else f"  {k:15s}: {v}")


print("Функции IC/ICIR (с Newey-West) загружены.")
print("Использование:")
print('  ic = compute_ic_series(df, "factor", "fwd_return", "date")')
print('  plot_ic_analysis(ic, nw_lags=5)')


In [ ]:
# === QUINTILE ANALYSIS + TURNOVER ===

def quintile_analysis(df_q, factor_col, return_col, date_col=None,
                      n_quantiles=5, annualize_factor=252):
    """
    Квинтильный анализ фактора + turnover.

    annualize_factor — 252 для дневных, 12 для месячных, 52 для недельных.
    """
    df_q = df_q.copy()

    if date_col is not None:
        # Cross-sectional: ранжируем внутри каждой даты
        df_q["quantile"] = df_q.groupby(date_col)[factor_col].transform(
            lambda x: pd.qcut(x.rank(method="first"), n_quantiles,
                              labels=range(1, n_quantiles+1), duplicates="drop")
        )
    else:
        df_q["quantile"] = pd.qcut(
            df_q[factor_col].rank(method="first"),
            n_quantiles, labels=range(1, n_quantiles+1), duplicates="drop"
        )

    # Доходности по квинтилям
    quint_returns = df_q.groupby("quantile")[return_col].agg(["mean", "std", "count"])
    quint_returns["sharpe_per_period"] = quint_returns["mean"] / quint_returns["std"]
    quint_returns["sharpe_annualized"] = (
        quint_returns["sharpe_per_period"] * np.sqrt(annualize_factor)
    )
    quint_returns.index = [f"Q{i}" for i in quint_returns.index]

    # Spread Q5 - Q1
    spread_mean = quint_returns["mean"].iloc[-1] - quint_returns["mean"].iloc[0]
    spread_ann  = spread_mean * annualize_factor

    print(f"Квинтильный анализ: {factor_col} → {return_col}")
    print(quint_returns.to_string())
    print(f"\nSpread Q5-Q1 (per period): {spread_mean:.4f}")
    print(f"Spread Q5-Q1 (annualized): {spread_ann:.4f}")

    # График
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    colors = ["coral" if r < 0 else "steelblue" for r in quint_returns["mean"]]
    axes[0].bar(quint_returns.index, quint_returns["mean"], color=colors, alpha=0.85)
    axes[0].axhline(0, color="black", linewidth=0.8)
    axes[0].set_title(f"Средняя доходность по квинтилям ({factor_col})")
    axes[0].set_ylabel("Доходность (per period)")

    axes[1].bar(quint_returns.index, quint_returns["sharpe_annualized"],
                color="steelblue", alpha=0.85)
    axes[1].axhline(0, color="black", linewidth=0.8)
    axes[1].set_title("Annualized Sharpe по квинтилям")
    axes[1].set_ylabel("Sharpe (annualized)")

    plt.tight_layout()
    plt.show()

    # Turnover — только для temporal
    if date_col is not None:
        turnover = compute_turnover(df_q, factor_col, date_col, n_quantiles)
        print(f"\nTurnover (доля активов меняющих Q1/Q5 между периодами):")
        print(f"  Q1: {turnover['Q1']:.2%}")
        print(f"  Q5: {turnover['Q5']:.2%}")
        if turnover["Q5"] > 0.5:
            print("  ⚠️  Высокий turnover — transaction costs будут значимы")

    return quint_returns


def compute_turnover(df_q, factor_col, date_col, n_quantiles=5):
    """Сколько активов (в среднем) покидают Q1 и Q5 между соседними датами."""
    # ID активов — нужна колонка типа asset_id. Используй своё имя.
    # Здесь — упрощённый вариант по строкам, требует заменить на реальный asset_id.
    asset_col = "asset_id"  # <-- поменяй на своё имя
    if asset_col not in df_q.columns:
        return {"Q1": np.nan, "Q5": np.nan}

    df_q = df_q.sort_values([date_col, asset_col])
    dates = df_q[date_col].unique()
    turnovers = {"Q1": [], "Q5": []}

    for i in range(1, len(dates)):
        prev = df_q[df_q[date_col] == dates[i-1]].set_index(asset_col)["quantile"]
        curr = df_q[df_q[date_col] == dates[i]].set_index(asset_col)["quantile"]
        common = prev.index.intersection(curr.index)
        for q_label, q_num in [("Q1", 1), ("Q5", n_quantiles)]:
            prev_in_q = prev[common] == q_num
            curr_in_q = curr[common] == q_num
            if prev_in_q.sum() > 0:
                # доля тех кто был в Q и выпал
                exited = (prev_in_q & ~curr_in_q).sum() / prev_in_q.sum()
                turnovers[q_label].append(exited)

    return {k: np.mean(v) if v else np.nan for k, v in turnovers.items()}


print("Функция quintile_analysis загружена.")


---
## 12. Residual & Error Analysis

После обучения смотрим на ошибки — это находит баги и направления для улучшения.

### Что проверяем на residual plots

1. **Residuals vs Predicted** — должны быть "облаком" без тренда.
   - *Воронка* (heteroscedasticity) → дисперсия ошибок зависит от предсказания → MSE/OLS дают биас. Лечить: WLS, log-transform таргета, Huber loss.
   - *Кривая* → модель упустила нелинейность. Лечить: FE, GBM.
2. **Q-Q plot of residuals** — проверка нормальности остатков (важно для p-values в линейных моделях).
3. **Residuals vs каждая фича** — если есть паттерн → фича закодирована неправильно, или нужно взаимодействие.
4. **Distribution of residuals** — должна быть симметричной вокруг 0.

### Heteroscedasticity tests
- **Breusch-Pagan** — `p < 0.05` → гетероскедастичность
- **White test** — альтернатива

**Вопрос:** *"Почему остатки важнее метрик типа R²?"*
→ R² = одно число, скрывает паттерны. Две модели с R² = 0.5 могут иметь абсолютно разные residual patterns — одна underfit, другая biased на экстремумах.


In [ ]:
# === RESIDUAL ANALYSIS ===
# Берём лучшую модель (например Ridge)
best_model = ridge  # или выбери любую
y_pred_train = best_model.predict(X_train_proc)
y_pred_test  = best_model.predict(X_test_proc)

residuals_train = y_train - y_pred_train
residuals_test  = y_test  - y_pred_test

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Residuals vs Predicted
axes[0, 0].scatter(y_pred_test, residuals_test, alpha=0.3, s=10, color="steelblue")
axes[0, 0].axhline(0, color="red", linestyle="--")
axes[0, 0].set_xlabel("Predicted")
axes[0, 0].set_ylabel("Residual")
axes[0, 0].set_title("Residuals vs Predicted (test)")

# 2. Q-Q plot of residuals
stats.probplot(residuals_test, dist="norm", plot=axes[0, 1])
axes[0, 1].set_title("Q-Q plot of residuals")

# 3. Гистограмма остатков
axes[1, 0].hist(residuals_test, bins=40, color="steelblue", edgecolor="white", alpha=0.8)
axes[1, 0].axvline(0, color="red", linestyle="--")
axes[1, 0].axvline(residuals_test.mean(), color="orange", linestyle="--",
                   label=f"Mean={residuals_test.mean():.4f}")
axes[1, 0].set_title(f"Распределение остатков (test), skew={residuals_test.skew():.2f}")
axes[1, 0].legend()

# 4. Predicted vs Actual
axes[1, 1].scatter(y_test, y_pred_test, alpha=0.3, s=10, color="steelblue")
lims = [min(y_test.min(), y_pred_test.min()), max(y_test.max(), y_pred_test.max())]
axes[1, 1].plot(lims, lims, "r--", linewidth=1.5)
axes[1, 1].set_xlabel("Actual")
axes[1, 1].set_ylabel("Predicted")
axes[1, 1].set_title("Predicted vs Actual")

plt.tight_layout()
plt.show()

# Heteroscedasticity — Breusch-Pagan (работает на train residuals)
try:
    from statsmodels.stats.diagnostic import het_breuschpagan
    bp_stat, bp_p, _, _ = het_breuschpagan(
        residuals_train,
        sm.add_constant(X_train_proc)
    )
    print(f"\nBreusch-Pagan: stat={bp_stat:.2f}, p={bp_p:.4g}")
    print(f"  → Гетероскедастичность {'обнаружена' if bp_p < 0.05 else 'не обнаружена'}")
except Exception as e:
    print(f"BP test failed: {e}")


In [ ]:
# === ERROR ANALYSIS — где модель ошибается больше всего? ===
# Смотрим на худшие 5% предсказаний — что в них общего?
test_df_err = X_test.copy()
test_df_err["y_true"]   = y_test.values
test_df_err["y_pred"]   = y_pred_test
test_df_err["abs_err"]  = (test_df_err["y_true"] - test_df_err["y_pred"]).abs()

worst = test_df_err.nlargest(max(10, len(test_df_err) // 20), "abs_err")
best  = test_df_err.nsmallest(max(10, len(test_df_err) // 20), "abs_err")

print("Топ-5 худших предсказаний:")
print(worst[["y_true", "y_pred", "abs_err"]].head().to_string())

# Сравниваем распределения признаков в best vs worst
numeric_err_cols = X_test.select_dtypes(include=np.number).columns[:8]
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()

for i, col in enumerate(numeric_err_cols):
    axes[i].hist(best[col].dropna(),  bins=20, alpha=0.5, label="best 5%",  color="steelblue")
    axes[i].hist(worst[col].dropna(), bins=20, alpha=0.5, label="worst 5%", color="coral")
    axes[i].set_title(col, fontsize=9)
    axes[i].legend(fontsize=7)

plt.suptitle("Распределения признаков: best vs worst predictions", y=1.02)
plt.tight_layout()
plt.show()


---
## 13. Robustness Checks

Это то, за что квант-интервьюеры ставят плюсы отдельно. Одноразовое "модель работает" — не доказательство.

### Что проверяем

1. **Subperiod stability** — работает ли модель одинаково в разных периодах времени?
2. **Sensitivity to hyperparameters** — сколько метрика меняется при ±20% от α, learning rate, depth?
3. **Random seed variance** — насколько результат зависит от seed? (>5% разницы — тревожно)
4. **Multiple testing correction** — если тестировали 20 факторов, FDR (Benjamini-Hochberg).
5. **Out-of-sample длительностью ≥ in-sample** — классическая рекомендация, лучше 2:1.
6. **Deflated Sharpe Ratio** — поправка Sharpe на multiple testing (Bailey & López de Prado).

### Multiple testing: почему Bonferroni — плохо в квантах

Bonferroni: `α_adj = α / m`. Слишком консервативен при коррелированных тестах. 
Benjamini-Hochberg (FDR) контролирует ожидаемую долю ложных открытий → более мягкий и адекватный для факторного исследования.


In [ ]:
# === SUBPERIOD STABILITY ===
# Делим test на 3 части по времени, смотрим метрики в каждой.
def subperiod_stability(y_true, y_pred, n_periods=3):
    idx = np.array_split(np.arange(len(y_true)), n_periods)
    results = []
    for i, p in enumerate(idx):
        rmse = np.sqrt(mean_squared_error(y_true.iloc[p], y_pred[p]))
        r2   = r2_score(y_true.iloc[p], y_pred[p])
        results.append({"period": f"P{i+1}", "n": len(p), "RMSE": rmse, "R2": r2})
    df_sub = pd.DataFrame(results)
    print("Subperiod stability:")
    print(df_sub.to_string(index=False))

    rmse_cv = df_sub["RMSE"].std() / df_sub["RMSE"].mean()
    print(f"\nCV of RMSE across periods: {rmse_cv*100:.1f}%")
    if rmse_cv > 0.3:
        print("  ⚠️  Нестабильность — модель может не работать в других режимах рынка")
    return df_sub


subperiod_stability(y_test, y_pred_test, n_periods=3)


In [ ]:
# === SENSITIVITY TO HYPERPARAMETERS ===
# Насколько результат зависит от alpha?
alphas_sens = [0.01, 0.1, 1.0, 10.0, 100.0]
sens_results = []
for a in alphas_sens:
    m = Ridge(alpha=a)
    m.fit(X_train_proc, y_train)
    rmse = np.sqrt(mean_squared_error(y_test, m.predict(X_test_proc)))
    sens_results.append({"alpha": a, "RMSE_test": rmse})

sens_df = pd.DataFrame(sens_results)
print("Ridge sensitivity to alpha:")
print(sens_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogx(sens_df["alpha"], sens_df["RMSE_test"], "o-", color="steelblue")
ax.set_xlabel("alpha (log scale)")
ax.set_ylabel("Test RMSE")
ax.set_title("Sensitivity: RMSE vs alpha")
plt.tight_layout()
plt.show()


In [ ]:
# === RANDOM SEED VARIANCE (для стохастических моделей) ===
seeds = [1, 7, 42, 100, 2024]
seed_rmses = []
for s in seeds:
    m = lgb.LGBMRegressor(
        n_estimators=300, learning_rate=0.05, num_leaves=63,
        subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
        random_state=s, verbose=-1
    )
    m.fit(X_train_proc, y_train)
    rmse = np.sqrt(mean_squared_error(y_test, m.predict(X_test_proc)))
    seed_rmses.append(rmse)

print(f"LightGBM RMSE across {len(seeds)} seeds:")
print(f"  Mean: {np.mean(seed_rmses):.4f}")
print(f"  Std:  {np.std(seed_rmses):.4f}")
print(f"  CV:   {np.std(seed_rmses) / np.mean(seed_rmses) * 100:.2f}%")

if np.std(seed_rmses) / np.mean(seed_rmses) > 0.05:
    print("  ⚠️  Высокая variance по seed — ансамблируй несколько моделей")


In [ ]:
# === MULTIPLE TESTING CORRECTION (Benjamini-Hochberg) ===
# Пример: тестируем 20 "факторов" (корреляций с таргетом), корректируем p-values.

# Искусственный пример — замени своим массивом p-values
np.random.seed(SEED)
# Симулируем: 15 шумовых + 5 реальных
p_values = np.concatenate([
    np.random.uniform(0, 1, 15),       # шум
    np.random.uniform(0, 0.03, 5)      # реальные сигналы
])
factor_names = [f"factor_{i}" for i in range(20)]

# Bonferroni (слишком строгий)
reject_bonf, p_bonf, _, _ = multipletests(p_values, alpha=0.05, method="bonferroni")

# BH (FDR — предпочтительный в квантах)
reject_bh, p_bh, _, _ = multipletests(p_values, alpha=0.05, method="fdr_bh")

mt_df = pd.DataFrame({
    "factor": factor_names,
    "p_raw":  p_values,
    "p_bonf": p_bonf,
    "p_BH":   p_bh,
    "sig_raw":  p_values < 0.05,
    "sig_bonf": reject_bonf,
    "sig_BH":   reject_bh,
}).sort_values("p_raw")

print("Multiple testing correction:")
print(mt_df.to_string(index=False))
print(f"\nSignificant at 5%:")
print(f"  Raw:        {mt_df['sig_raw'].sum()}  (ожидаем ~1 ложный из 20 шумовых)")
print(f"  Bonferroni: {mt_df['sig_bonf'].sum()}  (слишком строго)")
print(f"  BH (FDR):   {mt_df['sig_BH'].sum()}    (сбалансированно)")


In [ ]:
# === STATIONARITY CHECK (ADF test) для временных рядов ===
# H0: ряд нестационарный. p < 0.05 → стационарность.
# Нестационарные ряды (тренд, сезонность) ломают OLS — нужна дифференциация или detrend.

def adf_test(series, name="Series"):
    s = series.dropna()
    if len(s) < 20:
        print(f"{name}: мало данных для ADF")
        return
    result = adfuller(s, autolag="AIC")
    print(f"\n{name}:")
    print(f"  ADF stat: {result[0]:.4f}")
    print(f"  p-value:  {result[1]:.4g}")
    print(f"  Stationary: {'YES' if result[1] < 0.05 else 'NO (нужна дифференциация)'}")

# Пример для таргета
adf_test(y_train, "Target (train)")
# Для каждой числовой фичи
# for col in num_features[:3]:
#     adf_test(X_train[col], col)


---
## 14. Conclusions

Обязательно пиши выводы явно. Это показывает структурное мышление и уважение ко времени интервьюера.

### Структура вывода

1. **Data findings** — что нашёл в EDA (1-3 пункта)
2. **Best model + rationale** — какая выиграла и почему
3. **Limitations** — где модель может сломаться, что не проверено
4. **Next steps** — что бы исследовал при большем времени

### Частые "next steps" ответы

- Hyperparameter tuning через Optuna/Bayesian optimization
- Ensemble / stacking (Ridge + LGBM часто хорошо работают вместе)
- More feature engineering: interactions, lag features, rolling statistics
- Alternative loss functions (Huber для robust regression)
- Causal inference: если есть время — проверить confounders
- Deep learning: табличная задача + большой датасет → TabNet/FT-Transformer


In [ ]:
# === ФИНАЛЬНЫЙ SUMMARY ===
print("="*60)
print("SUMMARY")
print("="*60)

print("\n1. DATA")
print(f"   Rows: {df.shape[0]:,} | Columns: {df.shape[1]}")
print(f"   Avg missing: {df.isnull().mean().mean()*100:.1f}%")
print(f"   Target: {TARGET} (skew={df[TARGET].skew():.2f}, kurt={df[TARGET].kurt():.2f})")

print("\n2. MODELS — Test RMSE (lower is better)")
results_df_sorted = results_df.sort_values("RMSE_test")
for _, row in results_df_sorted.iterrows():
    marker = " <-- BEST" if row["RMSE_test"] == results_df_sorted["RMSE_test"].min() else ""
    print(f"   {row.name:20s}: RMSE={row['RMSE_test']:.4f}, "
          f"R²={row['R2_test']:+.4f}, gap={row['overfit_gap']:+.4f}{marker}")

best_name = results_df_sorted.index[0]
print(f"\n3. BEST MODEL: {best_name}")
print(f"   Test RMSE:    {results_df_sorted.iloc[0]['RMSE_test']:.4f}")
print(f"   Baseline RMSE: {results_df[results_df.index.str.contains('Baseline')]['RMSE_test'].min():.4f}")
improvement = (
    1 - results_df_sorted.iloc[0]["RMSE_test"] /
    results_df[results_df.index.str.contains("Baseline")]["RMSE_test"].min()
) * 100
print(f"   Improvement vs baseline: {improvement:.1f}%")

print("\n4. NEXT STEPS (if more time):")
print("   [ ] Hyperparameter tuning (Optuna, Bayesian)")
print("   [ ] Feature interactions + lag features")
print("   [ ] Ensemble (Ridge + LGBM stacking)")
print("   [ ] Residual analysis on worst predictions")
print("   [ ] Multiple testing correction for factor set")
print("   [ ] Test on subperiods — stability over time")
